In [1]:
!pip install pyspark


In [2]:
from google.colab import files
file=files.upload()

Saving carrier_master.csv to carrier_master.csv
Saving route_master.csv to route_master.csv
Saving shipments_75000.csv to shipments_75000.csv
Saving shipments_extra_12000.csv to shipments_extra_12000.csv
Saving warehouse_master.csv to warehouse_master.csv
Saving warehouse_sensor_config.json to warehouse_sensor_config.json


In [3]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('Practice_27th_October').getOrCreate()

In [4]:

shipments_df = spark.read.csv(
    "shipments_75000.csv",
    header=True,
    inferSchema=True
)

extra_df = spark.read.csv(
    "shipments_extra_12000.csv",
    header=True,
    inferSchema=True
)

warehouse_df = spark.read.csv(
    "warehouse_master.csv",
    header=True,
    inferSchema=True
)

carrier_df = spark.read.csv(
    "carrier_master.csv",
    header=True,
    inferSchema=True
)

route_df = spark.read.csv(
    "route_master.csv",
    header=True,
    inferSchema=True
)

sensor_df = spark.read.option(
    "multiLine", True
).json("warehouse_sensor_config.json")


In [5]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel
import time

PART A | DataFrame Operations

In [6]:
shipments_df.printSchema()
print("Count:",shipments_df.count())
shipments_df.show(15)

root
 |-- shipment_id: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- promised_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- carrier_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_weight_kg: double (nullable = true)
 |-- shipping_fee_inr: double (nullable = true)
 |-- shipping_cost_inr: double (nullable = true)
 |-- shipment_status: string (nullable = true)
 |-- delivery_attempts: integer (nullable = true)
 |-- damage_flag: string (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- handling_minutes: double (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- priority_flag: string (nullable = true)

Count: 75000
+-----------+------------+-------------+----------------------

In [8]:
shipments_df.select("shipment_id","warehouse_id","service_level","shipment_status","shipping_fee_inr").show()

+-----------+------------+-------------+---------------+----------------+
|shipment_id|warehouse_id|service_level|shipment_status|shipping_fee_inr|
+-----------+------------+-------------+---------------+----------------+
| SHP0000001|         W08|      Express|      Delivered|          554.34|
| SHP0000002|         W08|     Standard|      Delivered|          414.56|
| SHP0000003|         W01|      Express|      Delivered|          724.64|
| SHP0000004|         W03|     Standard|      Delivered|          287.86|
| SHP0000005|         W03|      Economy|      Delivered|          251.68|
| SHP0000006|         W01|      Express|      Delivered|           477.9|
| SHP0000007|         W02|     Standard|     In Transit|          298.68|
| SHP0000008|         W03|      Economy|     In Transit|          284.52|
| SHP0000009|         W02|     Standard|      Delivered|          861.74|
| SHP0000010|         W02|      Economy|      Delivered|          309.76|
| SHP0000011|         W04|     Standar

In [9]:
shipments_df.filter((col("service_level")=="Express")&(col("package_weight_kg")>5)).show()

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0000003|  2026-01-31|   2026-02-01|            2026-02-03| 

In [10]:
shipments_df.sort(col("shipping_fee_inr").desc()).show(20)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0044904|  2026-08-11|   2026-08-13|            2026-08-16| 

PART B | Aggregations and Joins

In [11]:
shipments_df.groupBy("shipment_status").count().show()

+---------------+-----+
|shipment_status|count|
+---------------+-----+
|       Returned| 6130|
|           Lost| 2241|
|     In Transit|10472|
|      Cancelled| 3723|
|      Delivered|52434|
+---------------+-----+



In [12]:

shipments_df.groupBy("service_level").agg(
    sum("shipping_fee_inr").alias("total_fees"),
    sum("shipping_cost_inr").alias("total_cost"),
    sum(
        when(
            col("shipping_cost_inr").isNotNull(),
            col("shipping_fee_inr") - col("shipping_cost_inr")
        )
    ).alias("total_profit"),
    sum(
        when(col("shipping_cost_inr").isNull(), 1)
        .otherwise(0)
    ).alias("missing_cost_count")
).withColumn(
    "profit_margin",
    when(
        col("total_fees") != 0,
        round(col("total_profit") / col("total_fees") * 100, 2)
    )
).show()


+-------------+--------------------+--------------------+------------------+------------------+-------------+
|service_level|          total_fees|          total_cost|      total_profit|missing_cost_count|profit_margin|
+-------------+--------------------+--------------------+------------------+------------------+-------------+
|      Express|1.0058468059999986E7|   7350805.699999983| 2514772.519999993|               401|         25.0|
|     Standard|1.3437372619999977E7|1.0110465900000025E7| 3043793.239999998|               844|        22.65|
|      Economy|   4489918.680000002|   3386212.500000007|1018119.2000000007|               258|        22.68|
+-------------+--------------------+--------------------+------------------+------------------+-------------+



In [15]:
shipments_df.join(warehouse_df,"warehouse_id","left")\
  .groupBy("region").agg(
      count("*").alias("no_of_shipments")
  ).show()

+------+---------------+
|region|no_of_shipments|
+------+---------------+
| South|          33018|
|  East|           5306|
|  West|          26909|
| North|           9767|
+------+---------------+



In [16]:
joined_df = shipments_df \
    .join(carrier_df, "carrier_id", "left") \
    .join(route_df, "route_id", "left")

joined_df.filter(
    col("shipping_cost_inr").isNotNull() &
    col("distance_km").isNotNull() &
    (col("distance_km") > 0)
).withColumn(
    "cost_per_km",
    col("shipping_cost_inr") / col("distance_km")
).groupBy(
    "carrier_id"
).agg(
    round(avg("cost_per_km"), 2).alias("avg_cost_per_km")
).show()


+----------+---------------+
|carrier_id|avg_cost_per_km|
+----------+---------------+
|       C03|           0.59|
|      NULL|           0.63|
|       C01|            0.6|
|       C06|           0.59|
|       C07|           0.61|
|       C05|           0.59|
|       C02|            0.6|
|       C04|           0.61|
+----------+---------------+



PART C | Null Handling and Conditional Logic

In [17]:
shipments_df.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c + "_missing")
    for c in [
        "carrier_id",
        "package_weight_kg",
        "shipping_cost_inr",
        "customer_rating"
    ]
]).show()


+------------------+-------------------------+-------------------------+-----------------------+
|carrier_id_missing|package_weight_kg_missing|shipping_cost_inr_missing|customer_rating_missing|
+------------------+-------------------------+-------------------------+-----------------------+
|               871|                     1129|                     1503|                  26255|
+------------------+-------------------------+-------------------------+-----------------------+



In [18]:
filled_df = shipments_df.fillna({
    "handling_minutes": 0
})

filled_df.show(5)

no_null_carrier_df = shipments_df.dropna(
    subset=["carrier_id"]
)

no_null_carrier_df.show(5)


+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0000001|  2026-05-24|   2026-05-24|            2026-05-26| 

In [19]:
weight_df = shipments_df.withColumn(
    "weight_category",
    when(col("package_weight_kg").isNull(), "Unknown")
    .when(col("package_weight_kg") < 2, "Light")
    .when(col("package_weight_kg") <= 10, "Medium")
    .otherwise("Heavy")
)

weight_df.select(
    "shipment_id",
    "package_weight_kg",
    "weight_category"
).show()


+-----------+-----------------+---------------+
|shipment_id|package_weight_kg|weight_category|
+-----------+-----------------+---------------+
| SHP0000001|             3.93|         Medium|
| SHP0000002|             5.62|         Medium|
| SHP0000003|             8.78|         Medium|
| SHP0000004|             0.97|          Light|
| SHP0000005|             NULL|        Unknown|
| SHP0000006|             6.05|         Medium|
| SHP0000007|             1.36|          Light|
| SHP0000008|             2.04|         Medium|
| SHP0000009|            18.73|          Heavy|
| SHP0000010|             2.02|         Medium|
| SHP0000011|             6.87|         Medium|
| SHP0000012|            11.83|          Heavy|
| SHP0000013|             3.73|         Medium|
| SHP0000014|             1.42|          Light|
| SHP0000015|             1.74|          Light|
| SHP0000016|             6.87|         Medium|
| SHP0000017|             2.58|         Medium|
| SHP0000018|             1.85|         

PART D | Date and Time Functions

In [20]:
date_df = shipments_df \
    .withColumn("booking_date", to_date("booking_date")) \
    .withColumn(
        "promised_delivery_date",
        to_date("promised_delivery_date")
    ) \
    .withColumn(
        "actual_delivery_date",
        to_date("actual_delivery_date")
    )
duration_df = date_df.filter(
    col("shipment_status") == "Delivered"
).withColumn(
    "delivery_duration",
    datediff(
        col("actual_delivery_date"),
        col("booking_date")
    )
)
duration_df.select(
    "shipment_id",
    "booking_date",
    "actual_delivery_date",
    "delivery_duration"
).show()


+-----------+------------+--------------------+-----------------+
|shipment_id|booking_date|actual_delivery_date|delivery_duration|
+-----------+------------+--------------------+-----------------+
| SHP0000001|  2026-05-24|          2026-05-27|                3|
| SHP0000002|  2026-05-17|          2026-05-21|                4|
| SHP0000003|  2026-01-31|          2026-02-04|                4|
| SHP0000004|  2026-03-12|          2026-03-18|                6|
| SHP0000005|  2026-08-16|          2026-08-29|               13|
| SHP0000006|  2026-02-06|          2026-02-09|                3|
| SHP0000009|  2026-08-12|          2026-08-21|                9|
| SHP0000010|  2026-06-01|          2026-06-11|               10|
| SHP0000011|  2026-04-24|          2026-05-01|                7|
| SHP0000013|  2026-06-26|          2026-06-29|                3|
| SHP0000014|  2026-04-07|          2026-04-09|                2|
| SHP0000016|  2026-06-23|          2026-06-27|                4|
| SHP00000

In [22]:
performance_df = date_df.withColumn(
    "delivery_performance",
    when(
        col("actual_delivery_date").isNull(),
        "Missing Actual Date"
    )
    .when(
        col("promised_delivery_date").isNull(),
        "Missing Promised Date"
    )
    .when(
        col("actual_delivery_date") < col("promised_delivery_date"),
        "Early"
    )
    .when(
        col("actual_delivery_date") == col("promised_delivery_date"),
        "On Time"
    )
    .otherwise("Late")
)
performance_df.select(
    "shipment_id",
    "delivery_performance"
).show()


+-----------+--------------------+
|shipment_id|delivery_performance|
+-----------+--------------------+
| SHP0000001|                Late|
| SHP0000002|             On Time|
| SHP0000003|                Late|
| SHP0000004|                Late|
| SHP0000005|                Late|
| SHP0000006|             On Time|
| SHP0000007| Missing Actual Date|
| SHP0000008| Missing Actual Date|
| SHP0000009|                Late|
| SHP0000010|                Late|
| SHP0000011|                Late|
| SHP0000012| Missing Actual Date|
| SHP0000013|             On Time|
| SHP0000014|             On Time|
| SHP0000015| Missing Actual Date|
| SHP0000016|               Early|
| SHP0000017|             On Time|
| SHP0000018|             On Time|
| SHP0000019|             On Time|
| SHP0000020|               Early|
+-----------+--------------------+
only showing top 20 rows


In [23]:
date_df.filter(
    col("shipment_status") == "Delivered"
).withColumn(
    "month",
    date_format("booking_date", "yyyy-MM")
).groupBy("month").agg(
    count("*").alias("delivered_count"),
    sum("shipping_fee_inr").alias("total_fees")
).orderBy("month").show()


+-------+---------------+------------------+
|  month|delivered_count|        total_fees|
+-------+---------------+------------------+
|2026-01|           6710| 2525138.440000005|
|2026-02|           6097|2268113.5000000037|
|2026-03|           6810|2530456.2400000026|
|2026-04|           6379|2389163.4600000014|
|2026-05|           6858|2549683.9999999963|
|2026-06|           6615|2447590.1200000006|
|2026-07|           6773|2512355.2199999997|
|2026-08|           6192|2306432.5199999996|
+-------+---------------+------------------+



PART E | Window Functions

In [24]:
region_df = shipments_df.join(
    warehouse_df,
    "warehouse_id",
    "left"
)
carrier_fees = region_df.groupBy(
    "region",
    "carrier_id"
).agg(
    sum("shipping_fee_inr").alias("total_fees")
)
w = Window.partitionBy("region").orderBy(
    col("total_fees").desc()
)
carrier_fees.withColumn(
    "rank",
    dense_rank().over(w)
).show()

+------+----------+------------------+----+
|region|carrier_id|        total_fees|rank|
+------+----------+------------------+----+
|  East|       C01|393862.61999999994|   1|
|  East|       C02|         373590.02|   2|
|  East|       C03|302260.92000000004|   3|
|  East|       C05|243322.91999999998|   4|
|  East|       C07|222025.85999999993|   5|
|  East|       C04|211583.64000000004|   6|
|  East|       C06| 195810.3399999999|   7|
|  East|      NULL|           24594.6|   8|
| North|       C02| 731442.0999999994|   1|
| North|       C01| 653223.6200000005|   2|
| North|       C03| 597883.4600000005|   3|
| North|       C05|464601.85999999975|   4|
| North|       C07|425396.69999999995|   5|
| North|       C04|393181.07999999984|   6|
| North|       C06|338296.45999999985|   7|
| North|      NULL|45235.700000000004|   8|
| South|       C02|2442652.5400000014|   1|
| South|       C01| 2271284.540000001|   2|
| South|       C03|1929008.1199999978|   3|
| South|       C07|1519441.49999

In [25]:
w = Window.partitionBy("warehouse_id").orderBy(
    col("booking_date").desc(),
    col("shipment_id").desc()
)
date_df.withColumn(
    "row_num",
    row_number().over(w)
).filter(
    col("row_num") == 1
).show()


+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+-------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|row_num|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+-------+
| SHP0072346|  2026-08-28|         NULL|

In [26]:
w = Window.partitionBy("warehouse_id") \
    .orderBy("booking_date", "shipment_id") \
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
date_df.withColumn(
    "running_total",
    sum("shipping_fee_inr").over(w)
).select(
    "warehouse_id",
    "shipment_id",
    "booking_date",
    "shipping_fee_inr",
    "running_total"
).show()


+------------+-----------+------------+----------------+------------------+
|warehouse_id|shipment_id|booking_date|shipping_fee_inr|     running_total|
+------------+-----------+------------+----------------+------------------+
|         W01| SHP0004062|  2026-01-01|          312.48|            312.48|
|         W01| SHP0006739|  2026-01-01|          227.02|             539.5|
|         W01| SHP0007553|  2026-01-01|          316.02|            855.52|
|         W01| SHP0008505|  2026-01-01|          572.68|1428.1999999999998|
|         W01| SHP0009674|  2026-01-01|           322.9|            1751.1|
|         W01| SHP0011312|  2026-01-01|           390.1|            2141.2|
|         W01| SHP0011588|  2026-01-01|          421.72|           2562.92|
|         W01| SHP0011698|  2026-01-01|          190.22|           2753.14|
|         W01| SHP0011973|  2026-01-01|          384.64|3137.7799999999997|
|         W01| SHP0017305|  2026-01-01|          203.48|3341.2599999999998|
|         W0

PART F | Union and Complex JSON

In [27]:
combined_df = shipments_df.unionByName(extra_df)
print("Combined records:", combined_df.count())

Combined records: 87000


In [28]:
combined_df.groupBy("shipment_id").count() \
    .filter(col("count") > 1).show()
unique_df = combined_df.dropDuplicates(["shipment_id"])
print("Before:", combined_df.count())
print("After:", unique_df.count())


+-----------+-----+
|shipment_id|count|
+-----------+-----+
| SHP0039848|    2|
| SHP0048411|    2|
| SHP0052790|    2|
| SHP0026413|    2|
| SHP0041611|    2|
| SHP0010090|    2|
| SHP0038340|    2|
| SHP0027530|    2|
| SHP0031755|    2|
| SHP0034259|    2|
| SHP0047068|    2|
| SHP0048280|    2|
| SHP0008086|    2|
| SHP0031960|    2|
| SHP0025784|    2|
| SHP0005611|    2|
| SHP0011388|    2|
| SHP0017357|    2|
| SHP0035268|    2|
| SHP0002969|    2|
+-----------+-----+
only showing top 20 rows
Before: 87000
After: 86250


In [31]:
sensor_result = sensor_df.withColumn(
    "sensor",
    explode("sensors")
)
sensor_result.select(
    "warehouse_id",
    "environment",
    "sensors"
).show(truncate=False)


+------------+--------------+------------------------------------------------------------------------------+
|warehouse_id|environment   |sensors                                                                       |
+------------+--------------+------------------------------------------------------------------------------+
|W01         |{45, {32, 18}}|[{false, W01-S1, Temperature}, {true, W01-S2, Humidity}, {true, W01-S3, Door}]|
|W01         |{45, {32, 18}}|[{false, W01-S1, Temperature}, {true, W01-S2, Humidity}, {true, W01-S3, Door}]|
|W01         |{45, {32, 18}}|[{false, W01-S1, Temperature}, {true, W01-S2, Humidity}, {true, W01-S3, Door}]|
|W02         |{50, {32, 18}}|[{true, W02-S1, Temperature}, {true, W02-S2, Humidity}, {true, W02-S3, Door}] |
|W02         |{50, {32, 18}}|[{true, W02-S1, Temperature}, {true, W02-S2, Humidity}, {true, W02-S3, Door}] |
|W02         |{50, {32, 18}}|[{true, W02-S1, Temperature}, {true, W02-S2, Humidity}, {true, W02-S3, Door}] |
|W03         |{55, 

PART G | Lazy Execution, Caching, and Persistence

In [32]:
pipeline_df = shipments_df \
    .filter(col("shipment_status") == "Delivered") \
    .join(warehouse_df, "warehouse_id", "left") \
    .withColumn(
        "profit",
        col("shipping_fee_inr") - col("shipping_cost_inr")
    ) \
    .groupBy("region") \
    .agg(sum("profit").alias("total_profit")) \
    .orderBy(col("total_profit").desc())
pipeline_df.explain()
pipeline_df.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_profit#1294 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_profit#1294 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=1522]
      +- HashAggregate(keys=[region#94], functions=[sum(profit#1293)])
         +- Exchange hashpartitioning(region#94, 200), ENSURE_REQUIREMENTS, [plan_id=1519]
            +- HashAggregate(keys=[region#94], functions=[partial_sum(profit#1293)])
               +- Project [region#94, (shipping_fee_inr#28 - shipping_cost_inr#29) AS profit#1293]
                  +- BroadcastHashJoin [warehouse_id#22], [warehouse_id#91], LeftOuter, BuildRight, false
                     :- Project [warehouse_id#22, shipping_fee_inr#28, shipping_cost_inr#29]
                     :  +- Filter (isnotnull(shipment_status#30) AND (shipment_status#30 = Delivered))
                     :     +- FileScan csv [warehouse_id#22,shipping_fee_inr#28,shipping_cost_inr#29,shipment_status#30] Batched: fal

In [33]:
delivered_df = shipments_df.filter(
    col("shipment_status") == "Delivered"
)
delivered_df.cache()
start = time.time()
delivered_df.count()
print("First count:", time.time() - start)
start = time.time()
delivered_df.count()
print("Second count:", time.time() - start)
start = time.time()
delivered_df.groupBy("service_level").count().show()
print("First aggregation:", time.time() - start)
start = time.time()
delivered_df.groupBy("service_level").count().show()
print("Second aggregation:", time.time() - start)
delivered_df.unpersist()


First count: 2.58776593208313
Second count: 0.21448254585266113
+-------------+-----+
|service_level|count|
+-------------+-----+
|      Express|14819|
|     Standard|28159|
|      Economy| 9456|
+-------------+-----+

First aggregation: 0.37827134132385254
+-------------+-----+
|service_level|count|
+-------------+-----+
|      Express|14819|
|     Standard|28159|
|      Economy| 9456|
+-------------+-----+

Second aggregation: 0.26598167419433594


DataFrame[shipment_id: string, booking_date: date, dispatch_date: date, promised_delivery_date: date, actual_delivery_date: date, warehouse_id: string, carrier_id: string, route_id: string, customer_segment: string, service_level: string, package_weight_kg: double, shipping_fee_inr: double, shipping_cost_inr: double, shipment_status: string, delivery_attempts: int, damage_flag: string, customer_rating: double, handling_minutes: double, payment_mode: string, priority_flag: string]

In [34]:
express_df = shipments_df.filter(
    col("service_level") == "Express"
)
express_df.persist(StorageLevel.MEMORY_AND_DISK)
print(express_df.count())
express_df.groupBy("shipment_status").count().show()
print(express_df.storageLevel)
express_df.unpersist()


21090
+---------------+-----+
|shipment_status|count|
+---------------+-----+
|       Returned| 1767|
|           Lost|  575|
|     In Transit| 2944|
|      Cancelled|  985|
|      Delivered|14819|
+---------------+-----+

Disk Memory Serialized 1x Replicated


DataFrame[shipment_id: string, booking_date: date, dispatch_date: date, promised_delivery_date: date, actual_delivery_date: date, warehouse_id: string, carrier_id: string, route_id: string, customer_segment: string, service_level: string, package_weight_kg: double, shipping_fee_inr: double, shipping_cost_inr: double, shipment_status: string, delivery_attempts: int, damage_flag: string, customer_rating: double, handling_minutes: double, payment_mode: string, priority_flag: string]

PART H | Partitioning and Optimization

In [35]:
df8 = shipments_df.repartition(8)
print("Partitions:", df8.rdd.getNumPartitions())
df8.withColumn(
    "partition_id",
    spark_partition_id()
).groupBy("partition_id").count() \
    .orderBy("partition_id").show()

Partitions: 8
+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 9375|
|           1| 9374|
|           2| 9376|
|           3| 9375|
|           4| 9375|
|           5| 9375|
|           6| 9375|
|           7| 9375|
+------------+-----+



In [36]:
df12 = shipments_df.repartition(12)
df_coalesce = df12.coalesce(3)
df_repartition = df12.repartition(3)
print("Original:", df12.rdd.getNumPartitions())
print("Coalesce:", df_coalesce.rdd.getNumPartitions())
print("Repartition:", df_repartition.rdd.getNumPartitions())
print("Coalesce execution plan:")
df_coalesce.explain()
print("Repartition execution plan:")
df_repartition.explain()


Original: 12
Coalesce: 3
Repartition: 3
Coalesce execution plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- Coalesce 3
      +- ShuffleQueryStage 0
         +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=2150]
            +- FileScan csv [shipment_id#17,booking_date#18,dispatch_date#19,promised_delivery_date#20,actual_delivery_date#21,warehouse_id#22,carrier_id#23,route_id#24,customer_segment#25,service_level#26,package_weight_kg#27,shipping_fee_inr#28,shipping_cost_inr#29,shipment_status#30,delivery_attempts#31,damage_flag#32,customer_rating#33,handling_minutes#34,payment_mode#35,priority_flag#36] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/shipments_75000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<shipment_id:string,booking_date:date,dispatch_date:date,promised_delivery_date:date,actual...
+- == Initial Plan ==
   Coalesce 3
   

In [37]:
output_df = shipments_df.join(
    warehouse_df,
    "warehouse_id",
    "left"
)
output_df.repartition(4).write \
    .mode("overwrite") \
    .parquet("/content/logistics_parquet")
import os
print(os.listdir("/content/logistics_parquet"))


['part-00001-67502bef-2c4d-4ba5-add1-2ce4856c5eba-c000.snappy.parquet', '._SUCCESS.crc', '.part-00001-67502bef-2c4d-4ba5-add1-2ce4856c5eba-c000.snappy.parquet.crc', '.part-00002-67502bef-2c4d-4ba5-add1-2ce4856c5eba-c000.snappy.parquet.crc', '_SUCCESS', 'part-00000-67502bef-2c4d-4ba5-add1-2ce4856c5eba-c000.snappy.parquet', '.part-00000-67502bef-2c4d-4ba5-add1-2ce4856c5eba-c000.snappy.parquet.crc', 'part-00002-67502bef-2c4d-4ba5-add1-2ce4856c5eba-c000.snappy.parquet', 'part-00003-67502bef-2c4d-4ba5-add1-2ce4856c5eba-c000.snappy.parquet', '.part-00003-67502bef-2c4d-4ba5-add1-2ce4856c5eba-c000.snappy.parquet.crc']


FINAL CHALLENGE | Logistics Performance Pipeline

In [38]:

from pyspark.sql.functions import *
from pyspark.sql.window import Window
cost_median = shipments_df.approxQuantile(
    "shipping_cost_inr", [0.5], 0.01
)[0]
weight_median = shipments_df.approxQuantile(
    "package_weight_kg", [0.5], 0.01
)[0]
clean_df = shipments_df.fillna({
    "shipping_cost_inr": cost_median,
    "package_weight_kg": weight_median
})

final_df = clean_df \
    .join(warehouse_df, "warehouse_id", "left") \
    .join(carrier_df, "carrier_id", "left") \
    .join(route_df, "route_id", "left")

final_df = final_df \
    .withColumn(
        "booking_date",
        to_date("booking_date")
    ) \
    .withColumn(
        "actual_delivery_date",
        to_date("actual_delivery_date")
    ) \
    .withColumn(
        "promised_delivery_date",
        to_date("promised_delivery_date")
    )

final_df = final_df.withColumn(
    "delivery_duration",
    when(
        col("shipment_status") == "Delivered",
        datediff(
            col("actual_delivery_date"),
            col("booking_date")
        )
    )
)

final_df = final_df.withColumn(
    "late_delivery",
    when(
        col("actual_delivery_date").isNull() |
        col("promised_delivery_date").isNull(),
        lit(None).cast("integer")
    )
    .when(
        col("actual_delivery_date") >
        col("promised_delivery_date"),
        1
    )
    .otherwise(0)
)

final_df = final_df.withColumn(
    "shipment_profit",
    col("shipping_fee_inr") - col("shipping_cost_inr")
)

report_df = final_df.groupBy(
    "region",
    "carrier_id"
).agg(
    count("*").alias("total_shipments"),

    sum("shipping_fee_inr").alias("total_revenue"),

    sum("shipment_profit").alias("total_profit"),

    sum("late_delivery").alias("late_count"),

    count("late_delivery").alias("evaluated_deliveries")
)

report_df = report_df.withColumn(
    "late_percentage",
    when(
        col("evaluated_deliveries") > 0,
        round(
            col("late_count") /
            col("evaluated_deliveries") * 100,
            2
        )
    )
).withColumn(
    "on_time_percentage",
    when(
        col("evaluated_deliveries") > 0,
        round(100 - col("late_percentage"), 2)
    )
)
rank_window = Window.partitionBy("region").orderBy(
    col("on_time_percentage").desc_nulls_last()
)
report_df = report_df.withColumn(
    "carrier_rank",
    dense_rank().over(rank_window)
)

report_df.cache()
report_df.count()
report_df.show(truncate=False)
report_df.orderBy(
    "region",
    "carrier_rank"
).show(truncate=False)

report_df.write \
    .mode("overwrite") \
    .partitionBy("region") \
    .parquet("/content/final_logistics_report")

import os
print(os.listdir("/content/final_logistics_report"))
report_df.unpersist()


+------+----------+---------------+------------------+------------------+----------+--------------------+---------------+------------------+------------+
|region|carrier_id|total_shipments|total_revenue     |total_profit      |late_count|evaluated_deliveries|late_percentage|on_time_percentage|carrier_rank|
+------+----------+---------------+------------------+------------------+----------+--------------------+---------------+------------------+------------+
|South |C05       |3838           |1437480.9200000009|344350.02000000054|1084      |2740                |39.56          |60.44             |1           |
|South |C04       |3602           |1366979.28        |330416.8799999996 |1003      |2472                |40.57          |59.43             |2           |
|South |C01       |6075           |2271284.540000001 |541483.1399999985 |1724      |4207                |40.98          |59.02             |3           |
|South |C03       |5197           |1929008.1199999978|465875.31999999995|154

DataFrame[region: string, carrier_id: string, total_shipments: bigint, total_revenue: double, total_profit: double, late_count: bigint, evaluated_deliveries: bigint, late_percentage: double, on_time_percentage: double, carrier_rank: int]